# Mars Terrain Hazard Detection (revised)

Original notebook treated AI4Mars-style **region labels** as YOLO **boxes**.
This revision keeps that detection setup (so results stay comparable) but:

- isolates secrets
- discovers the real Roboflow folder layout
- drops empty classes
- validates / de-duplicates boxes
- splits with a rare-class constraint
- evaluates on the **held-out test** split
- prints metrics from the result objects, not hard-coded numbers

If the real goal is terrain *coverage* (soil / sand / bedrock), train `yolov8n-seg` instead of detect.

In [ ]:
# Cell 0 — setup (install + single config + imports)
# Responsibility: environment and constants only. No download, no training.
%pip install -q roboflow ultralytics scikit-learn pyyaml

from __future__ import annotations

import json
import os
import random
import shutil
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import yaml
from sklearn.model_selection import train_test_split

@dataclass(frozen=True)
class Config:
    workspace: str = "s-workspace-cfuov"
    project: str = "ai4mars-sbhtm"
    version: int = 1
    export_format: str = "coco-segmentation"
    raw_dir: Path = Path("/content/ai4Mars-1")
    yolo_dir: Path = Path("/content/mars_yolo")
    runs_dir: Path = Path("/content/runs")
    allowed_classes: tuple[str, ...] = ("bedrock", "big-rock", "sand", "soil", "terrain")
    train_ratio: float = 0.70
    val_ratio: float = 0.20
    test_ratio: float = 0.10
    seed: int = 42
    imgsz: int = 640
    epochs: int = 50
    batch: int = 16
    patience: int = 10
    min_box_norm: float = 1e-4
    # drop boxes that cover almost the whole image unless the class is terrain-like
    max_box_area: float = 0.98

CFG = Config()
random.seed(CFG.seed)
print("config ready", CFG.raw_dir, CFG.yolo_dir)

In [ ]:
# Cell 1 — auth (isolated)
# Responsibility: obtain a Roboflow key. Nothing else reads Colab secrets.

def get_roboflow_api_key() -> str:
    key = os.environ.get("ROBOFLOW_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("ROBOFLOW") or userdata.get("ROBOFLOW_API_KEY")
    except Exception as exc:
        raise RuntimeError(
            "Roboflow key not found. Set ROBOFLOW_API_KEY or Colab secret ROBOFLOW."
        ) from exc
    if not key:
        raise RuntimeError("Roboflow secret exists but is empty.")
    return key

# Probe only — do not print the key.
_ = get_roboflow_api_key()
print("auth: Roboflow key loaded")

In [ ]:
# Cell 2 — download (idempotent)
# Responsibility: fetch one dataset version to CFG.raw_dir.

def download_dataset(cfg: Config) -> Path:
    from roboflow import Roboflow

    marker = cfg.raw_dir / "_annotations.coco.json"
    nested_jsons = list(cfg.raw_dir.glob("*/_annotations.coco.json"))
    if cfg.raw_dir.exists() and (marker.exists() or nested_jsons):
        print(f"reuse existing dataset at {cfg.raw_dir}")
        return cfg.raw_dir

    rf = Roboflow(api_key=get_roboflow_api_key())
    project = rf.workspace(cfg.workspace).project(cfg.project)
    version = project.version(cfg.version)
    dataset = version.download(
        cfg.export_format,
        location=str(cfg.raw_dir),
        overwrite=True,
    )
    print("downloaded to", dataset.location)
    return Path(dataset.location)

RAW = download_dataset(CFG)

In [ ]:
# Cell 3 — inspect raw export
# Responsibility: describe what Roboflow actually wrote. Do not assume train/valid/test.

def find_coco_jsons(root: Path) -> list[Path]:
    return sorted(root.rglob("*_annotations.coco.json")) + sorted(root.rglob("*.json"))

def inspect_raw(root: Path) -> list[Path]:
    jsons = []
    seen = set()
    for path in root.rglob("*.json"):
        if path.name.startswith("_annotations") or path.name.endswith(".coco.json"):
            if path not in seen:
                jsons.append(path)
                seen.add(path)
    if not jsons:
        jsons = [p for p in root.rglob("*.json")]

    print(f"root={root}")
    for dirpath, dirnames, filenames in os.walk(root):
        rel = os.path.relpath(dirpath, root)
        n_img = sum(1 for f in filenames if f.lower().endswith((".jpg", ".jpeg", ".png")))
        n_json = sum(1 for f in filenames if f.endswith(".json"))
        print(f"  {rel}: {n_img} images, {n_json} json, dirs={dirnames}")

    for jp in jsons:
        with jp.open() as f:
            coco = json.load(f)
        cats = [c.get("name") for c in coco.get("categories", [])]
        print(
            f"{jp.relative_to(root)}: images={len(coco.get('images', []))} "
            f"anns={len(coco.get('annotations', []))} cats={cats}"
        )
    if not jsons:
        raise FileNotFoundError(f"No COCO json under {root}")
    return jsons

COCO_JSONS = inspect_raw(CFG.raw_dir)

In [ ]:
# Cell 4 — merge COCO + convert to YOLO detect labels
# Responsibility: label conversion and split ownership. Training must not remap classes.

def load_merged_coco(json_paths: list[Path]) -> dict[str, Any]:
    merged = {"images": [], "annotations": [], "categories": []}
    seen_img = set()
    seen_ann = set()
    cats_by_id: dict[int, dict] = {}
    for jp in json_paths:
        with jp.open() as f:
            coco = json.load(f)
        parent = jp.parent
        for cat in coco.get("categories", []):
            cats_by_id[cat["id"]] = cat
        for img in coco.get("images", []):
            if img["id"] in seen_img:
                continue
            fname = img["file_name"]
            candidates = [parent / fname, parent / Path(fname).name, CFG.raw_dir / fname]
            src = next((p for p in candidates if p.exists()), None)
            if src is None:
                print("WARN missing image", fname)
                continue
            img = dict(img)
            img["_src_path"] = str(src)
            merged["images"].append(img)
            seen_img.add(img["id"])
        for ann in coco.get("annotations", []):
            key = (ann.get("image_id"), ann.get("id"))
            if key in seen_ann:
                continue
            merged["annotations"].append(ann)
            seen_ann.add(key)
    merged["categories"] = [cats_by_id[i] for i in sorted(cats_by_id)]
    return merged


def bbox_from_ann(ann: dict, img_w: int, img_h: int) -> tuple[float, float, float, float] | None:
    bbox = ann.get("bbox")
    if bbox and len(bbox) == 4:
        x, y, w, h = map(float, bbox)
    else:
        segs = ann.get("segmentation") or []
        xs, ys = [], []
        if isinstance(segs, list) and segs and isinstance(segs[0], list):
            for poly in segs:
                if len(poly) < 6:
                    continue
                xs.extend(poly[0::2])
                ys.extend(poly[1::2])
        if not xs:
            return None
        x, y = min(xs), min(ys)
        w, h = max(xs) - x, max(ys) - y

    x = max(0.0, min(x, img_w))
    y = max(0.0, min(y, img_h))
    w = max(0.0, min(w, img_w - x))
    h = max(0.0, min(h, img_h - y))
    if w < 1 or h < 1:
        return None
    return x, y, w, h


def to_yolo_line(box: tuple[float, float, float, float], img_w: int, img_h: int, cls_id: int, cfg: Config) -> str | None:
    x, y, w, h = box
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    bw = w / img_w
    bh = h / img_h
    if min(cx, cy, bw, bh) < 0 or max(cx, cy) > 1:
        return None
    bw = min(bw, 1.0)
    bh = min(bh, 1.0)
    if bw < cfg.min_box_norm or bh < cfg.min_box_norm:
        return None
    if bw * bh > cfg.max_box_area:
        return None
    return f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"


def build_class_map(coco: dict, allowed: tuple[str, ...]) -> tuple[dict[int, int], list[str]]:
    name_by_id = {c["id"]: c["name"] for c in coco["categories"]}
    counts: Counter[str] = Counter()
    for ann in coco["annotations"]:
        name = name_by_id.get(ann["category_id"])
        if name in allowed:
            counts[name] += 1
    # keep only classes that actually appear; preserve allowed order
    names = [n for n in allowed if counts[n] > 0]
    if not names:
        raise ValueError(f"No annotations in {allowed}. counts={dict(counts)} all={name_by_id}")
    coco_to_yolo = {}
    for cid, name in name_by_id.items():
        if name in names:
            coco_to_yolo[cid] = names.index(name)
    print("kept classes:", names, "counts:", {n: counts[n] for n in names})
    dropped = [n for n in allowed if counts[n] == 0]
    if dropped:
        print("dropped empty classes:", dropped)
    return coco_to_yolo, names


def labels_for_image(img: dict, anns: list[dict], coco_to_yolo: dict[int, int], cfg: Config) -> list[str]:
    w, h = int(img["width"]), int(img["height"])
    lines = []
    seen = set()
    for ann in anns:
        if ann["category_id"] not in coco_to_yolo:
            continue
        box = bbox_from_ann(ann, w, h)
        if box is None:
            continue
        line = to_yolo_line(box, w, h, coco_to_yolo[ann["category_id"]], cfg)
        if line and line not in seen:
            seen.add(line)
            lines.append(line)
    return lines


def stratified_split(items: list[dict], cfg: Config) -> dict[str, list[dict]]:
    # stratify on whether the rarest class is present, not on full class set
    rare_flags = [it["_rare"] for it in items]
    try:
        train, temp = train_test_split(
            items, test_size=(1 - cfg.train_ratio), random_state=cfg.seed, stratify=rare_flags
        )
        val_share = cfg.val_ratio / (cfg.val_ratio + cfg.test_ratio)
        temp_flags = [it["_rare"] for it in temp]
        valid, test = train_test_split(
            temp, test_size=(1 - val_share), random_state=cfg.seed, stratify=temp_flags
        )
    except ValueError:
        print("WARN: stratification failed (too few rare samples); using random split")
        train, temp = train_test_split(items, test_size=(1 - cfg.train_ratio), random_state=cfg.seed)
        val_share = cfg.val_ratio / (cfg.val_ratio + cfg.test_ratio)
        valid, test = train_test_split(temp, test_size=(1 - val_share), random_state=cfg.seed)
    return {"train": train, "valid": valid, "test": test}


def write_yolo_dataset(coco: dict, cfg: Config) -> tuple[Path, list[str], dict]:
    if cfg.yolo_dir.exists():
        shutil.rmtree(cfg.yolo_dir)

    coco_to_yolo, names = build_class_map(coco, cfg.allowed_classes)
    anns_by_img: dict[Any, list] = defaultdict(list)
    for ann in coco["annotations"]:
        anns_by_img[ann["image_id"]].append(ann)

    records = []
    skipped = 0
    for img in coco["images"]:
        lines = labels_for_image(img, anns_by_img[img["id"]], coco_to_yolo, cfg)
        if not lines:
            skipped += 1
            continue
        cls_ids = {int(line.split()[0]) for line in lines}
        rare_name = "big-rock"
        rare_id = names.index(rare_name) if rare_name in names else -1
        rec = {
            "img": img,
            "lines": lines,
            "cls_ids": cls_ids,
            "_rare": int(rare_id in cls_ids),
        }
        records.append(rec)
    print(f"images with usable boxes: {len(records)} (skipped {skipped})")

    splits = stratified_split(records, cfg)
    split_counts = {}
    for split, recs in splits.items():
        img_dir = cfg.yolo_dir / "images" / split
        lbl_dir = cfg.yolo_dir / "labels" / split
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        class_inst = Counter()
        for rec in recs:
            img = rec["img"]
            src = Path(img["_src_path"])
            dst = img_dir / src.name
            shutil.copy2(src, dst)
            (lbl_dir / f"{src.stem}.txt").write_text("\n".join(rec["lines"]) + "\n")
            for line in rec["lines"]:
                class_inst[names[int(line.split()[0])]] += 1
        split_counts[split] = {"images": len(recs), "instances": dict(class_inst)}
        print(f"{split}: {len(recs)} images, instances={dict(class_inst)}")

    yaml_path = cfg.yolo_dir / "data.yaml"
    payload = {
        "path": str(cfg.yolo_dir),
        "train": "images/train",
        "val": "images/valid",
        "test": "images/test",
        "nc": len(names),
        "names": names,
    }
    yaml_path.write_text(yaml.safe_dump(payload, sort_keys=False))
    print("wrote", yaml_path)
    return yaml_path, names, split_counts


COCO = load_merged_coco(COCO_JSONS)
YAML_PATH, CLASS_NAMES, SPLIT_COUNTS = write_yolo_dataset(COCO, CFG)
print("NOTE: converting region masks to boxes is a task change, not a lossless export.")

In [ ]:
# Cell 5 — train one architecture
# Responsibility: fit weights. Does not invent metrics.

from ultralytics import YOLO


def train_yolo(arch: str, cfg: Config, yaml_path: Path):
    run_name = f"mars_{arch}"
    model = YOLO(f"{arch}.pt")
    results = model.train(
        data=str(yaml_path),
        epochs=cfg.epochs,
        imgsz=cfg.imgsz,
        batch=cfg.batch,
        patience=cfg.patience,
        project=str(cfg.runs_dir),
        name=run_name,
        exist_ok=True,
        seed=cfg.seed,
        plots=True,
        verbose=True,
    )
    best = cfg.runs_dir / run_name / "weights" / "best.pt"
    if not best.exists():
        raise FileNotFoundError(f"training finished but {best} is missing")
    print(f"{arch} best weights -> {best}")
    return best, results


BEST_N, TRAIN_N = train_yolo("yolov8n", CFG, YAML_PATH)

In [ ]:
# Cell 6 — optional second architecture (same data, same seed)
BEST_S, TRAIN_S = train_yolo("yolov8s", CFG, YAML_PATH)

In [ ]:
# Cell 7 — evaluate on val AND held-out test
# Responsibility: read metrics from Ultralytics, never hard-code them.

def summarize(model_path: Path, yaml_path: Path, split: str) -> dict:
    model = YOLO(str(model_path))
    res = model.val(data=str(yaml_path), split=split, plots=True, verbose=True)
    names = res.names if isinstance(res.names, dict) else {i: n for i, n in enumerate(res.names)}
    per_class = {}
    maps = getattr(res.box, "maps", None)
    if maps is not None:
        for i, ap in enumerate(maps):
            per_class[names.get(i, str(i))] = {
                "mAP50-95": float(ap),
            }
    # class-wise mAP50 if present
    ap50 = getattr(res.box, "ap50", None)
    if ap50 is not None:
        for i, val in enumerate(ap50):
            per_class.setdefault(names.get(i, str(i)), {})["mAP50"] = float(val)
    out = {
        "model": str(model_path),
        "split": split,
        "precision": float(res.box.mp),
        "recall": float(res.box.mr),
        "mAP50": float(res.box.map50),
        "mAP50-95": float(res.box.map),
        "speed_ms_inference": float(res.speed.get("inference", float("nan"))),
        "per_class": per_class,
        "weight_mb": model_path.stat().st_size / 1e6,
    }
    return out


METRICS = {
    "yolov8n_val": summarize(BEST_N, YAML_PATH, "val"),
    "yolov8n_test": summarize(BEST_N, YAML_PATH, "test"),
    "yolov8s_val": summarize(BEST_S, YAML_PATH, "val"),
    "yolov8s_test": summarize(BEST_S, YAML_PATH, "test"),
}
print(json.dumps(METRICS, indent=2))

In [ ]:
# Cell 8 — training artifacts + qualitative predictions
from IPython.display import Image, display
import cv2
import matplotlib.pyplot as plt

for run_name in ("mars_yolov8n", "mars_yolov8s"):
    result_dir = CFG.runs_dir / run_name
    print("===", run_name, "===")
    for fname in ("results.png", "confusion_matrix.png"):
        path = result_dir / fname
        if path.exists():
            display(Image(filename=str(path)))
        else:
            print("missing", path)

test_dir = CFG.yolo_dir / "images" / "test"
test_imgs = [p for p in test_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
if not test_imgs:
    raise FileNotFoundError(f"no test images in {test_dir}")
k = min(6, len(test_imgs))
sample = random.sample(test_imgs, k)
model = YOLO(str(BEST_N))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flatten(), sample):
    plotted = model(str(img_path), verbose=False)[0].plot()
    ax.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
    ax.set_title(img_path.name[:40])
    ax.axis("off")
for ax in axes.flatten()[k:]:
    ax.axis("off")
fig.suptitle("YOLOv8n predictions on held-out test images")
fig.tight_layout()
out_png = Path("/content/detection_results.png")
fig.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()
print("saved", out_png)

In [ ]:
# Cell 9 — report generated from METRICS (no literals from a previous run)

def fmt(x):
    return f"{x:.3f}" if isinstance(x, float) else str(x)

n_test = METRICS["yolov8n_test"]
s_test = METRICS["yolov8s_test"]
n_val = METRICS["yolov8n_val"]
s_val = METRICS["yolov8s_val"]

print("=" * 60)
print("Mars Terrain Hazard Detection — computed report")
print("=" * 60)
print("splits:", json.dumps(SPLIT_COUNTS, indent=2))
print()
print(f"{'metric':<18} {'n val':>8} {'n test':>8} {'s val':>8} {'s test':>8}")
for key in ("mAP50", "mAP50-95", "precision", "recall", "speed_ms_inference"):
    print(f"{key:<18} {fmt(n_val[key]):>8} {fmt(n_test[key]):>8} {fmt(s_val[key]):>8} {fmt(s_test[key]):>8}")
print(f"{'weight_mb':<18} {n_val['weight_mb']:.2f}{'':>6} {s_val['weight_mb']:.2f}")

print("\nper-class mAP50 on TEST (YOLOv8n)")
for cls, vals in sorted(n_test["per_class"].items(), key=lambda kv: -kv[1].get("mAP50", 0)):
    ap = vals.get("mAP50")
    if ap is None:
        continue
    bar = "█" * int(ap * 20)
    print(f"{cls:<12} {ap:.3f} {bar}")

print("\nconstraints / caveats")
print("- big-rock image counts must be read from SPLIT_COUNTS, not from val() 'Images' column")
print("- val() during training is not an unseen test; use yolov8n_test / yolov8s_test")
print("- region-to-box conversion under-represents area hazards and over-penalizes large polygons")

In [ ]:
# Cell 10 — optional artifact export (does not fail the experiment if Drive is absent)

def export_artifacts():
    targets = [
        Path("/content/detection_results.png"),
        CFG.yolo_dir / "data.yaml",
        BEST_N,
        BEST_S,
    ]
    dest_root = None
    if Path("/content/drive/MyDrive").exists():
        dest_root = Path("/content/drive/MyDrive/mars_yolo_export")
    else:
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive")
            dest_root = Path("/content/drive/MyDrive/mars_yolo_export")
        except Exception as exc:
            print("Drive not available, skip export:", exc)
            return
    dest_root.mkdir(parents=True, exist_ok=True)
    for src in targets:
        if src.exists():
            shutil.copy2(src, dest_root / src.name)
            print("copied", src.name)
        else:
            print("skip missing", src)

export_artifacts()